# 02. Mineração de Dados e Avaliação

Este notebook é dedicado à etapa de **Mineração de Dados e Avaliação** do projeto de Detecção de Anomalias nos gastos com o Cartão de Pagamento do Governo Federal (CPGF). Como a base não possui rótulos indicando fraudes conhecidas, a abordagem escolhida é a **Detecção de Outliers** utilizando técnicas de agrupamento não supervisionado.

## 1. Introdução e Carga de Dados

Nesta etapa, carregamos o conjunto de dados pré-processado (`cpgf_normalizado.csv`). Esses dados já passaram por rigorosos filtros qualitativos e transformações quantitativas. As variáveis estritamente numéricas de comportamento de gasto foram normalizadas pelo método Z-Score (StandardScaler) com o intuito de deixar todas as features na mesma escala, o que é fundamental para algoritmos baseados em distância (como K-Means e HDBSCAN).

A tarefa formal consiste em agrupar os CPFs com comportamentos normais em clusters coesos e identificar aqueles cujos padrões de distanciamento ou rarefação (baixa densidade) fujam à regra, sinalizando potenciais abusos. O índice original do dataframe (`CPF PORTADOR`) será restaurado e deve ser mantido para garantir a rastreabilidade posterior das anomalias.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import hdbscan
import os

# Configurando o estilo visual das plotagens
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

# Caminho dos dados importados do pré-processamento
data_path = '../dados/cpgf_normalizado.csv'

# Carga dos dados garantindo a restauração do índice original (CPF PORTADOR)
df = pd.read_csv(data_path, index_col=0)

# Verificando as dimensões e os dados importados
display(df.head())
print(f"Dimensões do dataset de análise: {df.shape}")

## 2. O Baseline: K-Means

O **K-Means** é um tradicional e largamente disseminado algoritmo particional baseado em protótipos. Ele efetua a segmentação dos dados visando descobrir $K$ centróides que minimizem iterativamente a Soma dos Erros Quadrados (SSE) entre os pontos avaliados e o centróide designativo de pertencimento.

Apesar de possuir convergência otimizada computacionalmente, ele embasa seu particionamento em uma pressuposição estrita: os grupamentos têm matrizes isométricas e forma esférica perante um limiar isotrópico no subespaço de análise. Consequentemente, forçará pontos em grupos desproporcionais, gerando muitas vezes fatiamentos de conjuntos mais densos.

A estratégia de **Detecção de Outlier** consistirá na estipulação do nível de dispersão: após a convergência do algoritmo, extrairemos a **distância Euclidiana** de cada portador de cartão da sua respectiva assinatura matricial (o centróide). Instâncias apresentando um distanciamento profundo constituirão de comportamentos excêntricos, logo, nossos Outliers prospectos no baseline.

### 2.1. Escolha do K (Método do Cotovelo)

O Método do Cotovelo (*Elbow Method*) é um gráfico heurístico pautado em executar o K-Means variando o K. Nosso foco analítico encontra-se no mapeamento da inércia em busca da taxa marginal decrescente (o cotovelo da encosta), representando ganho inexpressivo de variância interna retida em troco de sobrepartição desnecessária do nosso baseline.

In [ ]:
# Definindo o intervalo para K de acordo com as diretrizes do notebook
k_range = range(2, 11)
sse = []

# Computando a soma dos erros quadráticos (inertia_)
for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans_temp.fit(df)
    sse.append(kmeans_temp.inertia_)

# Plotando graficamente o Método do Cotovelo
plt.figure()
plt.plot(k_range, sse, marker='o', linestyle='-', linewidth=2, markersize=8)
plt.title('Método do Cotovelo para Escolha de K Ótimo', fontsize=14)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Soma dos Erros Quadrados (SSE)')
plt.xticks(k_range)
plt.show()

### 2.2. Treinamento do Modelo K-Means

Fixaremos arbitrariamente o cômputo do baseline na parametrização $K=4$, simulando macroscopicamente uma divisão representacional nas despesas da governança (como baixo gasto, médio estável, alto moderado e extremos voláteis).

Após a partição, mediremos o vetor de distanciamento até o centro geográfico alocado como métrica contínua substituta formal do nível de excêntridade comportamental.

In [ ]:
# Parâmetro global com K=4 segundo a avaliação arquitetural
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')

# Estipulamos explicitamente a criação da coluna de cluster 
df['Cluster_KMeans'] = kmeans.fit_predict(df)

# Procedimento do Pipeline de Outlier: Distanciamento Geométrico
centroides = kmeans.cluster_centers_
distancias = []

# Isolando as métricas por CPF (Iterativo de cálculo para transparência da matriz)
for i in range(len(df)):
    # Acessaremos estritamente a linha (ignorando o cluster_kmeans predito)
    cluster_idx = df['Cluster_KMeans'].iloc[i]
    ponto = df.drop(columns=['Cluster_KMeans']).iloc[i].values
    centroide = centroides[cluster_idx]
    
    # Norm L2 (Euclidiana)
    dist = np.linalg.norm(ponto - centroide)
    distancias.append(dist)

df['Distancia_Centroide_KMeans'] = distancias

# Plotagem distributiva das anomalias percebidas no modelo em histograma contínuo
plt.figure()
sns.histplot(df['Distancia_Centroide_KMeans'], bins=50, kde=True, color='crimson')
plt.title('Dispersão do Distanciamento Centroidal (Isolamento de Outliers)', fontsize=14)
plt.xlabel('Distância Euclidiana (Magnitude de Excentricidade)')
plt.ylabel('Densidade Frequencial')
plt.show()

print("Top 5 gastos atípicos com mais severo distanciamento da norma central (K-Means):")
display(df.sort_values('Distancia_Centroide_KMeans', ascending=False).head())

## 3. O Modelo Avançado: HDBSCAN

Superado o agrupamento basal distanciacional forçando clusters estáticos universais, mergulhamos no patamar analítico de altíssimo rigor topológico: o modelo hierárquico espacial de base densidade.

O K-Means demonstra fadiga colossal falhando em clusters contínuos alongados ou irregulares (formatos arbitrários intrínsecos no real mundo governamental). Já o agrupamento estritamente densidade precursor (o clássico DBSCAN), naufraga ante matrizes públicas portando **densidades heterogêneas severas**: um Ministério super ativo opera massiva verba gerando uma alta densidade de proximidade analítica transacional, concomitantemente a órgãos pontuais isolucionistas atuando à margem em baixa densidade extrema. O raio fixo ($\\\epsilon$) único exterminaria ou fundiria indevidamente ambos na ótica de densidade global única do antigo algorítmo.

Contornando isso fenomenalmente, entrava o **HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise)**. Ele elimina o calcanhar de Aquiles do raio fixo de distância unificada $\\\epsilon$, introduzindo um construto hierárquico através de um corte por estabilidade (*cluster stability/lifetime*) no dendrograma das Árvores Geradoras Mínimas. Permitindo, então, densidades desiguais em paralelo com sucesso.

Sua aplicação central na Deteção de Fraude baseia-se num subproduto natural de excelência de seu código: uma quantia avulsa de portadores esparsos, rarefeitos à ponto de inexistirem subjacentes a quaisquer polos orgânicos de adensamento matricial, cairão renegados em rótulo categórico singular de **ruído (`-1`)**. Da mesma forma, disporemos da extração matemática vetorial `outlier_scores_`: uma inferência contínua de similaridade probabilística computada por proximidade condicional em tempo de execução validando o distanciamento isolacionista analítico no limite da anomalia.

In [ ]:
# Instanciando a topologia HDBSCAN
# min_cluster_size=15 estipula que o modelo desconsidere grupamentos extremamente pequenos
# min_samples=5 parametriza a vizinhança na contabilidade da densidade do nó
# gen_min_span_tree=True invoca a plotagem latente do MST (Minimum Spanning Tree)
modelo_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    gen_min_span_tree=True
)

# Desassociamento imperativo do subespaço (Revertemos às originais features apenas)
features_X = df.drop(columns=['Cluster_KMeans', 'Distancia_Centroide_KMeans'])

# Agrupamento
modelo_hdbscan.fit(features_X)

# Acolhendo métricas do artefato HDBSCAN finalizado no Dataset de Exportação
df['Cluster_HDBSCAN'] = modelo_hdbscan.labels_
df['HDBSCAN_Outlier_Score'] = modelo_hdbscan.outlier_scores_

# Levantamento dos rótulos-ruído (Outliers primários classificados como -1)
ruido_abs = len(df[df['Cluster_HDBSCAN'] == -1])
clusters_ativos = df['Cluster_HDBSCAN'].nunique() - 1

print(f"Topologia do HDBSCAN inferida (Clusters estruturais detectados): {clusters_ativos}")
print(f"Espaço rotulado puramente como ruído espacial (label -1): {ruido_abs} observações ({ruido_abs/len(df):.2%}).\n")

# Dispersando graficamente a densidade contínua geracional da Anomalia associativa
plt.figure()
sns.histplot(df['HDBSCAN_Outlier_Score'], bins=50, kde=True, color='indigo')
plt.title('Distribuição Contínua do Probabilistic Outlier Score (HDBSCAN)', fontsize=14)
plt.xlabel('Grau Probabilístico de Excentricidade / Ruído')
plt.ylabel('Frequência de Identificação')
plt.show()

print("Top 5 anomalias severas via score estatístico de Isolamento Geográfico (HDBSCAN):")
display(df.sort_values(by='HDBSCAN_Outlier_Score', ascending=False).head())

## 4. Avaliação Técnica

Ausentados estritamente na vida real a benesse perene de *Ground Truth Labels* em malhas de abuso antifraudes corporativas, dependemos incondicionalmente num viés Não-Supervisionado de aferições e métricas internas extrínsecas espaciais. Destaca-se isolar aqui o grandioso **Coeficiente de Silhueta Global (Silhouette Score)**.

Sua computação atrelará um escalonamento analógico coeso transcorrente de $-1$ até o cume imexível da clareza espacial de $+1$, cotejando em uníssono uma métrica pareada de força e contorno sobre:
1. **A Coesão Endógena Interna ($a$)**: Restringe que as variâncias e os limites sejam minimizados internamente, o ente tem ser minimamente distante da média do subgrupo à qual pertence.
2. **A Separação Exógena Polar ($b$)**: Penaliza a confusão, requerendo que o ente possua valsa e clareamento distanciável imenso ao vizinho central de cluster diametralmente opositivo da encosta.

Exigimos contudo, atenção suprema na validação impetrada no silhuetamento do HDBSCAN. A inserção não tratada da amálgama caótica desvinculada classificada e jogada aos ruidosos esparsados do espaço como (`label: -1`) causaria um imenso estrago da avaliação intrínseca, derrubando o silhueta ao abismo. Desse forma submeteremos sua extração mitigatória estritamente sobre matrizes límpidamente isoladas no núcleo central modelante limpo.

In [ ]:
# Extraindo a raiz vetorial Z-Score pura isolada de resíduos avaliativos das células anteriores
X = df.drop(columns=['Cluster_KMeans', 'Distancia_Centroide_KMeans', 'Cluster_HDBSCAN', 'HDBSCAN_Outlier_Score']).values

# Efetuando o Silhouette Coefficient - Baseline KMeans clássico sobre 100% da Massa
silhueta_global_kmeans = silhouette_score(X, df['Cluster_KMeans'])
print(f"Coeficiente Silhueta Clássico Particionado do Baseline (K-Means): {silhueta_global_kmeans:.4f}")

# Filtrando o ruído sistêmico (-1) provocado na isolação do Agrupador Hierárquico HDBSCAN
mascara_filtragem_hdbscan = (df['Cluster_HDBSCAN'] != -1)

# Extração contornante isolando as camadas de Densidades Orgânicas da Amostra
X_purificado = X[mascara_filtragem_hdbscan]
labels_purificados = df.loc[mascara_filtragem_hdbscan, 'Cluster_HDBSCAN']

if labels_purificados.nunique() > 1:
    silhueta_limpa_hdbscan = silhouette_score(X_purificado, labels_purificados)
    print(f"Coeficiente Silhueta Avaliado sob Matrizes Densas (Clusters Válidos HDBSCAN): {silhueta_limpa_hdbscan:.4f}")
else:
    print("HDBSCAN gerou uma matriz insuficiente para travejamento da Silhueta Avaliativa.")

## 5. Exportação dos Resultados

Cimentamos integralmente aqui os frutos exatos e refinados resultantes dos compêndio basais aos avançados. O conjunto consolidatório conterá invariavelmente sua chave primária resguardada `CPF PORTADOR` no traçado originário dos índices, salvaguardando as *features* normativas primárias unidas num abraço colunar solidificado aos `outlier_scores_` espaciais densitários atípicos e suas medições intercentroidais distanciadas K-Means.

A extração `cpgf_minerado.csv` será insumo imprescindível para a última etapa da esteira principal: o **Pós-processamento de Insights**.

In [ ]:
# Caminho de consolidação arquitetural na saída avaliativa exportacional da persistência matricial
output_caminho = '../dados/cpgf_minerado.csv'

# Exportação direta atrelando indubitavelmente o índice restabelecido com êxito garantindo a rastreabilidade investigativa
df.to_csv(output_caminho)

print(f"Malha exportada com Sucesso para o endereço '{output_caminho}'.")
display(df.info())